# ViRead VietOCR V2 — chạy một job trong Colab miễn phí

Backend tự xếp job khi dataset sẵn sàng. Khi muốn train, kết nối runtime Colab và chạy cell bên dưới. Cell chỉ nhận tối đa một job đang chờ rồi thoát; không poll nền và không cần nhập job ID. Sau khi train xong, nếu còn job khác thì chạy lại cell. GPU miễn phí tùy tình trạng cấp phát và runtime có thể ngắt.

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import subprocess, sys, os

def secret(name, required=True):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if required and not value:
        raise RuntimeError(f"Missing Colab Secret: {name}")
    return value

API_BASE = secret("VIREAD_API_BASE").rstrip("/")
WORKER_TOKEN = secret("VIREAD_TRAINING_WORKER_TOKEN")
REPOSITORY_REF = secret("VIREAD_REPOSITORY_REF", required=False) or "main"
REPOSITORY_URL = "https://github.com/trancuong3/capstone1.git"
REPO = Path("/content/viread-capstone1")
drive.mount("/content/drive")

if not REPO.exists():
    subprocess.run(["git", "clone", "--filter=blob:none", REPOSITORY_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", REPOSITORY_REF], check=True)
dirty = subprocess.run(["git", "-C", str(REPO), "status", "--porcelain"], check=True, capture_output=True, text=True).stdout.strip()
if dirty:
    raise RuntimeError("Worker source checkout is modified; use a clean runtime directory.")
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO / "ai/vietocr_v2/requirements.txt")], check=True)
sys.path.insert(0, str(REPO))
os.chdir(REPO)
from ai.vietocr_v2.colab_worker import run_next_job_once

PERSISTENT_WORK = Path("/content/drive/MyDrive/ViRead/VietOCR-V2-worker")
PERSISTENT_WORK.mkdir(parents=True, exist_ok=True)
print("Checking for one ready training job. Secret values will not be displayed.")
processed = run_next_job_once(API_BASE, WORKER_TOKEN, PERSISTENT_WORK)
if processed:
    print("One training job finished processing. Check ViRead Admin for its run, benchmark, and quality-gate status.")
else:
    print("No ready job was available. No training run was started.")
